<a href="https://colab.research.google.com/github/JordanTerwilliger/Intro-to-Deep-Learning/blob/main/HW1/HW1_CIFAR10_CNN_Q2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Jordan Terwilliger, 801343938, HW1

https://github.com/JordanTerwilliger/Intro-to-Deep-Learning

In [1]:
import numpy as np

import torch as torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt

In [2]:
torch.manual_seed(1)

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

train_data = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle = True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle = True, num_workers=2)

100%|██████████| 170M/170M [00:13<00:00, 12.8MB/s]


In [4]:
class_names = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck'] # CIFAR-10 image classes

VGG-11 has 5 vgg blocks for a a total of 2 + 6 conv layers and then 4096->4096->1000 Linear Layers. which means that if we scaled the 32x32 input to have 2 vgg blocks, each with 2 conv. layers since that was the main idea behind vgg net was stacking conv. layers, and then 41->41->10 Linear Layers. Total Param count would be


Input:  3x32x32
VGG Block 1 Output: 128x16x16 (2 Convs which start at 64 and double output channels but only halves resolution)
VGG Block 2 Output : 512x8x8 (2 convs which start at 128 and double output channels but only halves resolution)

512x8x8 -> 41FC -> 41FC -> 10FC

Approximately (512x8x8)+1 x 41 + (41+1 x 41) + (41+1 x10)= 33310 parameters

Choosing a more complex VGG to replicate would only increase parameter count so we must go with VGG-11

In [20]:
class ModifiedVGG(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(3,64,3, padding=1)
    self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
    self.pool = nn.MaxPool2d(2,2)
    self.conv3 = nn.Conv2d(128,256,3,padding=1)
    self.conv4 = nn.Conv2d(256,512,3, padding=1)
    self.fc1 = nn.Linear(512*8*8, 41)
    self.fc2 = nn.Linear(41,41)
    self.fc3 = nn.Linear(41,10)

  def forward(self,x):
    x = F.relu(self.conv1(x))
    x = F.relu(self.conv2(x))
    x = self.pool(x)
    x = F.relu(self.conv3(x))
    x = F.relu(self.conv4(x))
    x = self.pool(x)
    x = torch.flatten(x,1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = F.relu(self.fc3(x))
    return x

In [21]:
net  = ModifiedVGG()
loss = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr = 0.001, momentum = 0.9)

In [ ]:
epochs = 10
#Lists for storing loss and validation values
train_loss_list = []
val_loss_list = []
val_accuracy_list = []
for epoch in range(epochs):
  running_loss = 0.0
  net.train()
  for i, data in enumerate(train_loader):
    inputs, labels = data
    inputs = inputs
    labels = labels

    optimizer.zero_grad()

    outputs = net(inputs)

    loss_function = loss(outputs, labels)
    loss_function.backward()
    optimizer.step()

    running_loss += loss_function.item()

  correct = 0
  total = 0

  net.eval()
  with torch.no_grad():
    for data in test_loader:
      images, labels = data
      outputs = net(images)
      loss_function = loss(outputs, labels)
      running_loss += loss_function.item()
      _, predicted = torch.max(outputs.data, 1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()
  val_loss = running_loss / len(test_loader)
  val_loss_list.append(val_loss)
  val_accuracy = 100 * correct / total
  val_accuracy_list.append(val_accuracy)

  train_loss_list.append(running_loss/len(train_loader))
  print(f'Epoch: {epoch}, Loss: {running_loss / len(train_loader):.4f}, Validation Accuracy: {val_accuracy}, Validation Loss: {val_loss}')

In [ ]:
from torchsummary import summary
import torch

# Check if a GPU is available and move the model to it
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
net.to(device)
print(summary(net, input_size=(3,32,32)))